# 07 Charts

**Owner:** Each person their own functions

**What this notebook does:** collects all chart functions in one place and tests them. From here they are copied into the app.

**Reads:** the clean files

**Writes:** nothing (the functions go into the three `app/panels_*.py` files)

This is a main notebook. It must run from top to bottom with Kernel → Restart and Run All.
Write simple code in small steps: one operation per line, print the result after each step.

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW = "../data/raw/"
CLEAN = "../data/clean/"
pd.set_option("display.max_columns", 50)

In [ ]:
vdem_regime_spans = pd.read_csv(CLEAN + "vdem_regime_spans.csv")
ucdp_spans = pd.read_csv(CLEAN + "ucdp_spans.csv")
sipri_suppliers = pd.read_csv(CLEAN + "sipri_suppliers.csv")
panel = pd.read_csv(CLEAN + "panel.csv")
event_time = pd.read_csv(CLEAN + "event_time.csv")


## Same look everywhere (do this first)

Write the colour dictionaries here: `regime_colours` for the four regime labels and `intensity_colours` for "minor" and "war".
Every timeline function follows the rules in `CONVENTIONS.md`: `ax.set_xlim(1950, 2025)`, a short title, a y-label with the unit where there are numbers, and a dashed line at `year`.
Copy the dictionaries to the top of all three panels files too, right after `import pandas as pd` and `import matplotlib.pyplot as plt`.

In [ ]:
# your code here


regime_colours = {
    "closed autocracy": "#4C78A8",
    "electoral autocracy": "#F58518",
    "electoral democracy": "#54A24B",
    "liberal democracy": "#E45756",
}
intensity_colours = {"minor": "#F2C14E", "war": "#D62728"}

# Conflict and regime (Nicolas) → `app/panels_conflict.py`

## regime_bands(ax, spans, iso3, year=None)

**Owner:** Nicolas. Paste the working function from your sandbox. When the test below shows the right chart, copy the function unchanged into `app/panels_conflict.py`.

In [ ]:
# your code here


def regime_bands(ax, spans, iso3, year=None):
    rows = spans[spans["iso3"] == iso3].copy()
    for label, start, end in zip(rows["regime_label"], rows["start"], rows["end"]):
        ax.barh(0, end - start + 1, left=start, color=regime_colours[label])
    ax.set_xlim(1950, 2025)
    ax.set_yticks([])
    ax.set_title("Regime")
    if year is not None:
        ax.axvline(year, linestyle="--", color="grey")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 1.2))
regime_bands(ax, vdem_regime_spans, "UKR", year=2014)
plt.show()

## conflict_bars(ax, spans, iso3, year=None)

**Owner:** Nicolas. Paste the working function from your sandbox. When the test below shows the right chart, copy the function unchanged into `app/panels_conflict.py`.

In [ ]:
# your code here


def conflict_bars(ax, spans, iso3, year=None):
    intensity_names = {1: "minor", 2: "war"}
    rows = spans[spans["iso3"] == iso3].copy()
    if len(rows) == 0:
        ax.text(0.5, 0.5, "No conflict on record", transform=ax.transAxes, ha="center", va="center", color="grey")
    for intensity, start, end in zip(rows["max_intensity"], rows["start"], rows["end"]):
        label = intensity_names[intensity]
        ax.barh(0, end - start + 1, left=start, color=intensity_colours[label])
    ax.set_xlim(1950, 2025)
    ax.set_yticks([])
    ax.set_title("Conflict")
    if year is not None:
        ax.axvline(year, linestyle="--", color="grey")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 2.4))
conflict_bars(axes[0], ucdp_spans, "COL", year=1990)
conflict_bars(axes[1], ucdp_spans, "XXX")   # no data: should show a short message
plt.tight_layout()
plt.show()

# Arms (Yair) → `app/panels_arms.py`

## arms_line(ax, panel, iso3, year=None)

**Owner:** Yair. Paste the working function from your sandbox. When the test below shows the right chart, copy the function unchanged into `app/panels_arms.py`.

In [ ]:
def arms_line(ax, panel, iso3, year=None): 
    rows = panel[panel["iso3"] == iso3]
    ax.bar(rows["year"], rows["arms_imports_tiv"])
    ax.set_xlim(1950, 2025)
    ax.set_title("Arms imports")
    ax.set_ylabel("TIV, millions")
    if year is not None:
        ax.axvline(year, linestyle="--", color="grey")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 1.6))
arms_line(ax, panel, "IND", year=1999)
plt.show()

## supplier_bars(ax, suppliers, iso3, year=None)

**Owner:** Yair. Paste the working function from your sandbox. When the test below shows the right chart, copy the function unchanged into `app/panels_arms.py`.

In [ ]:
def supplier_bars(ax, suppliers, iso3, year=None):
    country = suppliers[suppliers["iso3"] == iso3]
    wide = country.pivot(index="year", columns="supplier", values="tiv").fillna(0)
    if wide.empty:
        ax.set_title(f"Suppliers of: {iso3}")
        ax.text(0.5, 0.5, "No arms imports recorded", transform=ax.transAxes, ha="center")
        return
    else:
        bottom = [0] * len(wide)
        for product in wide.columns:
            ax.bar(wide.index, wide[product], bottom=bottom, label=product)
            bottom = [b + v for b, v in zip(bottom, wide[product])]

        ax.set_title(f"Top suppliers of {iso3}")
        ax.set_ylabel("TIV, millions")
        ax.set_xlim(1950, 2025)
        if year is not None:
                ax.axvline(year, linestyle="--", color="grey")
        ax.legend(fontsize=7, loc="upper left" )

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 3.2))
supplier_bars(axes[0], sipri_suppliers, "IND", year=1999)
supplier_bars(axes[1], sipri_suppliers, "XXX")   # no data: should show a short message
plt.tight_layout()
plt.show()

## global_trends(ax, panel, year=None)

**Owner:** Yair. Paste the working function from your sandbox. When the test below shows the right chart, copy the function unchanged into `app/panels_arms.py`.

In [ ]:
from matplotlib.ticker import PercentFormatter

def global_trends(ax, panel, year=None):
    conflict_active = panel.groupby("year", as_index=True).agg(share=("conflict_active", "mean"))
    ax.plot(conflict_active)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    ax.set_xlim(1950, 2025)
    ax.set_title("Share of countries in conflict")
    ax.set_ylabel("share")
    ax.axvline(2000, linestyle="--", color="grey")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
global_trends(ax, panel, year=2000)
plt.show()

# Democracy and main result (Yair) → `app/panels_analysis.py`

## democracy_line(ax, panel, iso3, year=None)

**Owner:** Yair. Paste the working function from your sandbox. When the test below shows the right chart, copy the function unchanged into `app/panels_analysis.py`.

In [ ]:
def democracy_line(ax, panel, iso3, year=None):
    country = panel[panel["iso3"] == iso3]
    ax.plot(country["year"], country["democracy_index"])
    ax.set_ylim(0, 1)
    ax.set_xlim(1950, 2025)
    ax.set_title(f"Democracy index of: {iso3}")
    ax.set_ylabel("0 to 1")
    if year is not None:
        ax.axvline(year, linestyle="--", color="grey")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 1.6))
democracy_line(ax, panel, "UKR", year=2014)
plt.show()

## main_chart(ax, event_time)

**Owner:** Yair. Paste the working function from your sandbox. When the test below shows the right chart, copy the function unchanged into `app/panels_analysis.py`.

In [ ]:
def main_chart(ax, event_time):
    for group, times in event_time.groupby("importer_group"):
        ax.plot(times["t"], times["dem_change"], label=group)
    ax.axvline(0, color="grey")
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_xlabel("Years since conflict onset")
    ax.set_ylabel("Change in democracy index")
    ax.legend()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
main_chart(ax, event_time)
plt.show()

## Final look check

This is how the app's country page will look: five charts under each other sharing one year axis. Run it for UKR, then change UKR to CHE (no conflict) and run it again.

In [ ]:
fig, axes = plt.subplots(5, 1, sharex=True, figsize=(10, 8))
regime_bands(axes[0], vdem_regime_spans, "CHE", 2014)
conflict_bars(axes[1], ucdp_spans, "CHE", 2014)
democracy_line(axes[2], panel, "CHE", 2014)
arms_line(axes[3], panel, "CHE", 2014)
supplier_bars(axes[4], sipri_suppliers, "CHE", 2014)
plt.tight_layout()
plt.show()